In [ ]:
import sys
sys.path.append("../")


import torch
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# dep graph generation functions
def smart_connect(graph, prev, current):
    if isinstance(prev, (list, tuple, set)):
        for p in prev:
            graph.add_edge(p, current)
    else:
        graph.add_edge(prev, current)

def depg_gen_bottleneck(name, graph, prev, cfg, shortcut=True):
    input_nodes = prev
    for k, v in cfg.items():
        current = f'{name}.{k}'
        graph.add_node(current, value=v)
        smart_connect(graph, prev, current)
        prev = current
    if shortcut:
        return input_nodes + [prev] if isinstance(input_nodes, list) else [input_nodes, prev]
    return [prev]

def depg_gen_c2f(name, graph, prev, cfg, shortcut=True):
    cv2_prev_nodes = [] # prev nodes of cv2
    current = f'{name}.cv1'
    graph.add_node(current, value=cfg['cv1'])
    smart_connect(graph, prev, current)
    prev = current
    cv2_prev_nodes.append(prev) # since cv1 output, prev is not a list

    for key, mcfg in cfg.items():
        if 'm' in key:
            prev = depg_gen_bottleneck(f'{name}.{key}', graph, prev, mcfg, shortcut=shortcut)
            cv2_prev_nodes  = cv2_prev_nodes + prev

    current = f'{name}.cv2'
    graph.add_node(current, value=cfg['cv2'])
    smart_connect(graph, cv2_prev_nodes, current)
    prev = current
    return prev

def depg_gen_sppf(name, graph, prev, cfg):
    current = f'{name}.cv1'
    graph.add_node(current, value=cfg['cv1'])
    smart_connect(graph, prev, current)
    prev = current

    current = f'{name}.cv2'
    graph.add_node(current, value=cfg['cv2'])
    smart_connect(graph, prev, current)
    prev = current

    return prev

def depg_gen_detect(name, graph, prev, cfg):
    prev_copy = prev
    current = f'{name}.cv2.0'
    graph.add_node(current, value=cfg['cv2.0'])
    smart_connect(graph, prev, current)
    prev = current

    current = f'{name}.cv2.1'
    graph.add_node(current, value=cfg['cv2.1'])
    smart_connect(graph, prev, current)
    prev = current

    current = f'{name}.cv2.2'
    graph.add_node(current, value=cfg['cv2.2'])
    smart_connect(graph, prev, current)
    prev = current

    prev = prev_copy
    current = f'{name}.cv3.0'
    graph.add_node(current, value=cfg['cv3.0'])
    smart_connect(graph, prev, current)
    prev = current

    current = f'{name}.cv3.1'
    graph.add_node(current, value=cfg['cv3.1'])
    smart_connect(graph, prev, current)
    prev = current

    current = f'{name}.cv3.2'
    graph.add_node(current, value=cfg['cv3.2'])
    smart_connect(graph, prev, current)
    prev = current


In [ ]:
# metadata
layer_type_map = {
    'conv2d' : ['layer_0', 'layer_1', 'layer_3', 'layer_5', 'layer_7',
                'layer_16', 'layer_19'],
    'c2f' : ['layer_2', 'layer_4', 'layer_6', 'layer_8', 
            'layer_12', 'layer_15', 
            'layer_18', 'layer_21'],
    'sppf': ['layer_9'],
    'detect': ['layer_22'],
}

out2msin_layers = {
    # concat input
    'layer_4': 'layer_15',
    'layer_6': 'layer_12',
    'layer_12': 'layer_18',
    'layer_9': 'layer_21',
    # anchor input
    'layer_15': 'layer_22',
    'layer_18': 'layer_22',
}

shortcut_layers = ['layer_2', 'layer_4', 'layer_6', 'layer_8',]

In [ ]:
# build dependency graph

# load quantization config
checkpoint_path = '../checkpoints/qat_fixed.pt' 
loaded_cfg = torch.load(checkpoint_path, device, weights_only=False)['qcfg']

# directed graph
graph = nx.DiGraph()
ms_prevs = defaultdict(list)

prev = None
for layer, cfg in loaded_cfg.items():
    # check multi-scale input layer
    if layer in ms_prevs.keys():
        prev2 = ms_prevs[layer]
        prev = prev2 + [prev]

    if layer in layer_type_map['conv2d']:
        current = layer
        graph.add_node(current, value=cfg)
        if prev is not None:
            smart_connect(graph, prev, current)
        prev = current

    elif layer in layer_type_map['c2f']:
        if layer in shortcut_layers:
            shortcut = True
        else:
            shortcut = False
        prev = depg_gen_c2f(layer, graph, prev, cfg)

    elif layer in layer_type_map['sppf']:
        prev = depg_gen_sppf(layer, graph, prev, cfg)

    elif layer in layer_type_map['detect']:
        prev = depg_gen_detect(layer, graph, prev, cfg)

    # check whether the output of the layer goes into for multi-scale input layer
    if layer in out2msin_layers.keys():
        p = prev if isinstance(prev, list) else [prev]
        ms_prevs[out2msin_layers[layer]].extend(p)


In [ ]:
# visualize the dependency graph

# 각 노드의 depth 계산
depths = {}
def assign_depth(node, depth=0):
    depths[node] = depth
    for child in graph.successors(node):
        assign_depth(child, depth + 1)

# 루트 찾기 (in_degree == 0 인 노드)
roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
for root in roots:
    assign_depth(root)

# depth별로 노드 모으기 (삽입 순서 유지)
levels = defaultdict(list)
for node in graph.nodes():
    levels[depths[node]].append(node)

# 위치 계산
pos = {}
x_spacing = 1.2
y_spacing = 1.2

for depth, nodes_at_level in levels.items():
    n = len(nodes_at_level)
    for i, node in enumerate(nodes_at_level):
        x = (i - (n - 1) / 2) * x_spacing
        y = -depth * y_spacing
        pos[node] = (x, y)

# 간선 색 결정 ----------------------------------------
edge_colors = []
for u, v in graph.edges():
    # u의 next node(=successor)가 2개 이상이면 파란색, 아니면 검정
    if len(list(graph.successors(u))) >= 2:
        edge_colors.append("blue")
    else:
        edge_colors.append("gray")

# 시각화
plt.figure(figsize=(10, 8))

# 노드 그리기
nx.draw_networkx_nodes(
    graph, pos,
    node_color='lightblue',
    node_size=200,
    linewidths=0.5
)

# 라벨 그리기
nx.draw_networkx_labels(
    graph, pos,
    font_size=6
)

# 간선 그리기 (색 지정!)
nx.draw_networkx_edges(
    graph, pos,
    arrows=True,
    arrowstyle='-|>',
    arrowsize=4,
    edge_color=edge_colors,
    width=1
)

plt.title("Dataflow of YOLOv8n", fontsize=10)
plt.axis('off')
plt.show()


In [ ]:
# check whether quantization config is compatible for NPU model and generate NPU config
def rename_node(node):
    return node.replace('layer_', 'model.')

ncfg = {}
ncfg_flag=0

for node in graph.nodes:
    # sanity check: is output activation scale is consistent?
    output_act_scale = [graph.nodes[next_node]['value'][2] for next_node in list(graph.successors(node))]
    if len(set(output_act_scale)) > 1:
        print(f'\n--- Warning: inconsistent output activation scales at node {node}: {output_act_scale} @ node{list(graph.successors(node))}\n')
        ncfg_flag=1
    elif len(output_act_scale) > 1:
        print(f'\n+++ Note: multiple successors at node {node}: {output_act_scale}\n')
    try:
        next = list(graph.successors(node))[0]
        num_bits = graph.nodes[next]['value'][0]
        fy = graph.nodes[next]['value'][2]
        fx = graph.nodes[node]['value'][2]
        fw = graph.nodes[node]['value'][1]
    except:
        print(f'(end node)')
        num_bits = graph.nodes[node]['value'][0]
        fy = 3 # unsigned: 4 / signed: 3
        fx = graph.nodes[node]['value'][2]
        fw = graph.nodes[node]['value'][1]
    shift = fx + fw - fy
    print(f'node {node} - fw: {fw}, fx: {fx}, fy: {fy}, num_bits: {num_bits}, shift: {shift}')
    # print(f'node {rename_node(node)} - fw: {fw}, fx: {fx}, fy: {fy}, num_bits: {num_bits}, shift: {shift}')
    if 'model.22.cv2' in rename_node(node) or 'model.22.cv3' in rename_node(node):
        conv_name = rename_node(node)
        if conv_name[-1] == '2':
            ncfg[conv_name[:-1] + '0.' + conv_name[-1]] = {
                'fx': fx,
                'fw': fw,
                'fy': fy,
                'num_bits': num_bits,
                'shift': shift
            }
            ncfg[conv_name[:-1] + '1.' + conv_name[-1]] = {
                'fx': fx,
                'fw': fw,
                'fy': fy,
                'num_bits': num_bits,
                'shift': shift
            }
            ncfg[conv_name[:-1] + '2.' + conv_name[-1]] = {
                'fx': fx,
                'fw': fw,
                'fy': fy,
                'num_bits': num_bits,
                'shift': shift
            }
        else:
            ncfg[conv_name[:-1] + '0.' + conv_name[-1] + '.conv'] = {
                'fx': fx,
                'fw': fw,
                'fy': fy,
                'num_bits': num_bits,
                'shift': shift
            }
            ncfg[conv_name[:-1] + '1.' + conv_name[-1] + '.conv'] = {
                'fx': fx,
                'fw': fw,
                'fy': fy,
                'num_bits': num_bits,
                'shift': shift
            }
            ncfg[conv_name[:-1] + '2.' + conv_name[-1] + '.conv'] = {
                'fx': fx,
                'fw': fw,
                'fy': fy,
                'num_bits': num_bits,
                'shift': shift
            }
    else:
        ncfg[rename_node(node) + '.conv'] = {
            'fx': fx,
            'fw': fw,
            'fy': fy,
            'num_bits': num_bits,
            'shift': shift
        }
ncfg['model.22.dfl.conv.weight'] = {
    'fx': fy,
    'fw': 0,
    'fy': 3, # unsigned: 4 / signed: 3
    'num_bits': num_bits,
    'shift': fx + fw - fy
}
if ncfg_flag:
    print('\n>>> !! NPU quantization config generation failed due to incompatible quantization config.\n')
else:
    print('\n>>> NPU quantization config generated successfully.\n')
